--------------------------------------------------------------------------------------------
## Convolutional + attention model
- Using CNN and Trasnformers model

In [1]:
import os, re
import pickle
import numpy as np
import pandas as pd
from glob import glob
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import load_img , img_to_array
from tensorflow.keras.applications.vgg16 import VGG16 , preprocess_input
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.applications.densenet import DenseNet201
from tensorflow.keras.layers import Input , Dense , LSTM , Embedding , Dropout , add

from tensorflow.keras.layers import Add, Activation, RepeatVector, Concatenate
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, GlobalAveragePooling2D

import time
from datetime import timedelta as td

In [2]:
import tensorflow as tf

tf.config.list_physical_devices('GPU')
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

with tf.device('/GPU:0'):
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)



print(tf.test.is_built_with_cuda())
print(tf.test.is_gpu_available())

TensorFlow: 2.13.1
GPUs: []
False
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
False


In [3]:
import psutil
import GPUtil # Only for Nvidea

# Resources where the training has been made
mem = psutil.virtual_memory()
gpus = GPUtil.getGPUs()
print("CPU cores:", psutil.cpu_count(logical=True))
print('-----------------------------------------')
print(f"Total RAM: {mem.total / (1024**3):.2f} GB")
print(f"Used RAM: {mem.used / (1024**3):.2f} GB")
print(f"Available RAM: {mem.available / (1024**3):.2f} GB")
print('-----------------------------------------')
for gpu in gpus:
    print(f"GPU: {gpu.name}")
    print(f"Total memory: {gpu.memoryTotal}MB")

CPU cores: 8
-----------------------------------------
Total RAM: 31.71 GB
Used RAM: 9.61 GB
Available RAM: 22.10 GB
-----------------------------------------
GPU: NVIDIA GeForce RTX 2060
Total memory: 6144.0MB


In [4]:
# Start timer for the training
start_time = time.perf_counter()

In [5]:
# Paths inside Google cloud
BUCKET = "weather-description-ml-bucket-20250920-144834"
resources_path = os.path.join(f"gs://{BUCKET}", "resources")

# Local paths
resources_path = os.path.join('..', '..', 'resources')

path_openAI    = os.path.join(resources_path, 'utils', 'images_weather_description_openAI.csv')
path_ibericam  = os.path.join(resources_path, 'images_ibericam')
path_captions  = os.path.join(resources_path, 'captions.txt')
path_model     = os.path.join(resources_path, 'utils', 'best_model.h5')

### 1. Generate the mapping for captions in the dataset

In [6]:
df_openAI   = pd.read_csv(path_openAI)
list_images_ibercam = glob(os.path.join(path_ibericam, '*.jpg'))

# Delete this lines for working with the hole dataset
# df_openAI = df_openAI[df_openAI.path_image.isin(list_images_ibercam)].reset_index()

In [7]:
df_aux               = df_openAI[['path_image', 'description']]
df_aux               = df_aux.copy()  # Create a full copy to avoid the warning
df_aux['path_image'] = df_aux['path_image'].apply(lambda x: x.split(os.sep)[-1])
df_aux.columns       = ['image', 'caption']

# Get the name of the images
df_aux['image_name'] = df_aux["image"].str.split('.').str[0]

# Cleaning and formating the dataset before applying LSTM
df_aux['caption_mod'] = df_aux["caption"].apply(lambda caption: [f'startseq "{re.sub("[^A-Za-z . -]", "", caption.lower())}" endseq'])

# Creating a dictinary with the name of the photos and the captions
mapping_dic          = dict(zip(df_aux['image_name'], df_aux['caption_mod']))
list_captions        = df_aux['caption_mod'].values
list_captions        = [item[0] for item in list_captions]

### 2. Load CNN prebuild models
- Modify the preloaded models from keras to:
    - Inputs: mantains the original inputs of the original model
    - Outputs: Takes the second last layer of the model avoiding the last one to be used later as an input to the LSTM

In [8]:
##### 16 layer network using 3x3 convolutions image classification #####
model = VGG16()
##### Dense networks with direct connections among all the layers to improve the gradient flow #####
# model = DenseNet201()
##### Residual networks, prevents performance degradation #####
# model = ResNet152()

# model = Model(inputs = model.inputs , outputs = model.layers[-2].output)
desired_output = model.get_layer('fc2').output
model = Model(inputs=model.input, outputs=desired_output)

# print(model.summary())

### 3. Transform the images
- Open the images and adapt it to work with tensorflow

In [9]:
features = {}
list_images_ibercam = glob(os.path.join(path_ibericam, '*.jpg'))

for index, img_path in enumerate(tqdm(list_images_ibercam)):
    
    # load the image from file in a certain format adapted to the model size
    image = load_img(img_path, target_size= (224, 224))#(model.input.shape[1], model.input.shape[2]))
    
    # convert image pixels to numpy array
    image = img_to_array(image)
    
    # reshape data for model
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    
    # preprocess image for tensorflow
    image = preprocess_input(image)
    
    # extract features
    feature = model.predict(image, verbose=0)

    
    # Get image name
    image_id = os.path.splitext(os.path.basename(img_path))[0]
    
    # store feature
    features[image_id] = feature

# Store the features of the images in a pickle file (for saving time for running multiple experiments)
# pickle.dump(features, open(os.path.join(resources_path, 'features.pkl'), 'wb'))

  0%|          | 0/6982 [00:00<?, ?it/s]

### 6. Data generator

In [10]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0

    while True:
        for key in data_keys:
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                if len(seq) < 2:
                    continue

                in_seq = seq[:-1]
                out_seq = seq[1:]

                in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                out_seq = pad_sequences([out_seq], maxlen=max_length, padding='post')[0]

                out_seq = to_categorical(out_seq, num_classes=vocab_size)

                X1.append(features[key].reshape(-1))  # asegura dimensión correcta
                X2.append(in_seq)
                y.append(out_seq)

                n += 1
                if n == batch_size:
                    yield ((np.array(X1), np.array(X2)), np.array(y))
                    X1, X2, y = list(), list(), list()
                    n = 0


### 4. Tokenizer the text data

In [11]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(list_captions)
vocab_size = len(tokenizer.word_index) + 1

# Maximum size of one of the captions
max_length = max(len(caption.split()) for caption in list_captions)

### 5. Create LSTM model
- **shape=(4096,)** - output length of the features from the VGG model

- **Dense** - single dimension linear layer array

- **Dropout()** - used to add regularization to the data, avoiding over fitting & dropping out a fraction of the data from the layers

- **model.compile()** - compilation of the model

- **loss=’sparse_categorical_crossentropy’** - loss function for category outputs

- **optimizer=’adam’** - automatically adjust the learning rate for the model over the no. of epochs

- Model plot shows the concatenation of the inputs and outputs into a single layer

- Feature extraction of image was already done using VGG, no CNN model was needed in this step.

In [12]:
# Parameters
max_length = 50
vocab_size = 5000
embed_size = 256
num_heads = 8
ff_dim = 512  # Feed-forward dimension
dropout_rate = 0.3

# Positional Encoding (seno-coseno)
def positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angle_rads = pos * angle_rates
    pos_encoding = np.zeros((max_len, d_model))
    pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

pos_encoding = positional_encoding(max_length, embed_size)

# --- Image branch ---
inputs1 = Input(shape=(4096,))
x_img = Dropout(dropout_rate)(inputs1)
x_img = Dense(embed_size, activation='relu')(x_img)
x_img = Dropout(dropout_rate)(x_img)
x_img = RepeatVector(max_length)(x_img)  # (batch, seq_len, embed_size)

# --- Text branch ---
inputs2 = Input(shape=(max_length,))
x_txt = Embedding(vocab_size, embed_size, mask_zero=True)(inputs2)
x_txt = Add()([x_txt, pos_encoding])  # add positional encoding
x_txt = Dropout(dropout_rate)(x_txt)

# --- Transformer Decoder Block (can be stacked) ---
def transformer_decoder_block(x, context, num_heads, ff_dim, dropout_rate, name):
    # Self-Attention (on text)
    attn1 = MultiHeadAttention(num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate, name=f"{name}_selfattn")(x, x)
    out1 = Add()([x, attn1])
    out1 = LayerNormalization()(out1)

    # Cross-Attention (text attends to image)
    attn2 = MultiHeadAttention(num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate, name=f"{name}_crossattn")(out1, context, context)
    out2 = Add()([out1, attn2])
    out2 = LayerNormalization()(out2)

    # Feed Forward Network
    ff = Dense(ff_dim, activation='relu')(out2)
    ff = Dropout(dropout_rate)(ff)
    ff = Dense(embed_size)(ff)
    out3 = Add()([out2, ff])
    out3 = LayerNormalization()(out3)
    
    return out3

# Stack multiple decoder blocks
x = transformer_decoder_block(x_txt, x_img, num_heads, ff_dim, dropout_rate, name="decoder_block1")
x = transformer_decoder_block(x, x_img, num_heads, ff_dim, dropout_rate, name="decoder_block2")

# Output layer (per timestep)
outputs = Dense(vocab_size, activation='softmax')(x)

# --- Final Model ---
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 50)]                 0         []                            
                                                                                                  
 embedding (Embedding)       (None, 50, 256)              1280000   ['input_3[0][0]']             
                                                                                                  
 add (Add)                   (None, 50, 256)              0         ['embedding[0][0]']           
                                                                                                  
 input_2 (InputLayer)        [(None, 4096)]               0         []                            
                                                                                            

### 5. Prepare data for trainning
- Train split data

In [13]:
image_ids = list(mapping_dic.keys())
split = int(len(image_ids) * 0.90)
train = image_ids[:split]
test = image_ids[split:]

epochs = 50
batch_size = 32
steps = len(train) // batch_size

### 7. Train Model

In [ ]:
for i in range(epochs):
    print("Epoch", i , " in a total of", epochs)
    # create data generator
    generator = data_generator(train, mapping_dic, features, tokenizer, max_length, vocab_size, batch_size)
    # fit for one epoch
    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)

# Save the best model
model.save(path_model)

Epoch 0  in a total of 50
  3/196 [..............................] - ETA: 53:33 - loss: 8.4254 - accuracy: 0.0011    

### 8. Generate Captions for the image

In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [ ]:
def sample_prediction(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [ ]:
def predict_caption(model, image, tokenizer, max_length, temperature=1.0):
    # Add start sequence token
    in_text = 'startseq'

    for _ in range(max_length):
        # Convert text to sequence and pad
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')

        # Predict next word probabilities
        preds = model.predict([image, sequence], verbose=0)
        preds = preds[0, len(in_text.split())-1]

        # Apply temperature-based sampling for diversity
        preds = np.log(preds + 1e-10) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        # Sample next word index
        next_index = np.random.choice(len(preds), p=preds)

        # Map predicted index to word
        next_word = idx_to_word(next_index, tokeniz

        # End if endseq is predicted or invalid word
        if next_word is None or next_word == 'endseq':
            break

        # Append predicted word to sequence
        in_text += ' ' + next_word

    return in_text


### 9. Model Validation

In [ ]:
actual, predicted = list(), list()

for key in tqdm(test):
    # get actual caption
    captions = mapping_dic[key]
    # predict the caption for image
    y_pred = predict_caption(model, features[key], tokenizer, max_length)
    # split into words
    actual_captions = [caption.split() for caption in captions]
    y_pred = y_pred.split()
    # append to the list
    actual.append(actual_captions)
    predicted.append(y_pred)
# calcuate BLEU score
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))

In [ ]:
# Ends the timer for the training
end_time   = time.perf_counter()
total_time = str(td(seconds= int(end_time - start_time)))
print("The final training time has been: ", total_time)

## 10. Visualize the Results

In [ ]:
def generate_caption(img_path):
    image = Image.open(img_path)
    captions = mapping_dic[image_id]
    print('---------------------Actual---------------------')
    for caption in captions:
        print(caption)
    # predict the caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)
    print('--------------------Predicted--------------------')
    print(y_pred)
    plt.imshow(image)

In [ ]:
def generate_inference_metrics() :
    list_images_ibericam = glob(os.path.join(path_ibericam, '*.jpg'))
    cont = 0
    total_time_inf = 0
    
    for image_path in list_images_ibericam :
        if cont < 5 :
            start_time_inf = time.perf_counter()
            generate_caption(image_path)
            end_time_inf = time.perf_counter()
            total_time_inf += end_time_inf - start_time_inf
        else :
            break
            
        cont +=1
        total_time_inf = total_time_inf / 5
        print("--------------------------------------------------------------------------------------")
        print("--------------------------------------------------------------------------------------")
        print("The time per instance during the inference is an average of: ", str(td(seconds= int(total_time_inf))))

In [ ]:
generate_caption("../../resources/images_ibericam/albentosa-20230429-130501.jpg")

In [ ]:
generate_inference_metrics()